In [1]:
import pandas as pd
import os

def load_fixtures(data_folder='data'):
    gameweeks_per_season = {}
    for season in os.listdir(data_folder):
        season_path = os.path.join(data_folder, season)
        if not os.path.isdir(season_path):
            continue
            
        matches_per_gameweek = {}
        fixtures_path = os.path.join(season_path, 'fixtures.csv')
        try:
            fixtures_df = pd.read_csv(fixtures_path)
            for _, row in fixtures_df.iterrows():
                event = row['event']
                if event not in matches_per_gameweek:
                    matches_per_gameweek[event] = []
                matches_per_gameweek[event].append(row)
            gameweeks_per_season[season] = matches_per_gameweek
        except FileNotFoundError:
            print(f"Warning: fixtures.csv not found for season {season}")
    return gameweeks_per_season

In [7]:
def get_teams_for_season(season, data_folder='data'):
    """
    Try to load teams.csv from the season folder first.
    If not found, fall back to master_team_list.csv for that season.
    """
    # First, try to load from season folder
    teams_path = os.path.join(data_folder, season, 'teams.csv')
    try:
        teams_df = pd.read_csv(teams_path)
        return teams_df[['id', 'name']]
    except FileNotFoundError:
        # Fall back to master_team_list.csv
        try:
            master_teams = pd.read_csv('master_team_list.csv')

            # Normalize column names (lowercase, strip spaces)
            master_teams.columns = [c.strip().lower() for c in master_teams.columns]

            # Now we know the columns are: season, team, team_name
            season_teams = master_teams[master_teams['season'] == season].copy()

            if season_teams.empty:
                print(f"Warning: Season {season} not found in master_team_list.csv")
                return None
                
            # Rename to match fixtures.csv expectations:
            season_teams = season_teams.rename(columns={
                'team': 'id',
                'team_name': 'name'
            })

            return season_teams[['id', 'name']]

        except FileNotFoundError:
            print("Warning: master_team_list.csv not found")
            return None


In [8]:
def fix_opponent_and_difficulty(df, data_folder='data'):
    """
    Rebuild Opponent Name, Opponent ID, and Opponent Difficulty from fixtures
    """
    fixed_opponent_ids = []
    fixed_opponent_names = []
    fixed_opponent_difficulties = []
    fixed_is_home = []
    
    match_dictionary = load_fixtures(data_folder)
    
    # Cache teams data by season
    season_teams_cache = {}
    
    for idx, row in df.iterrows():
        if idx % 1000 == 0:  # Progress indicator
            print(f"Processing row {idx}/{len(df)}...")
            
        season = row['season']
        gameweek = row['Gameweek']
        player_team_name = row['Player Team Name']
        
        try:
            # Get teams data for this season (with caching)
            if season not in season_teams_cache:
                teams_df = get_teams_for_season(season, data_folder)
                season_teams_cache[season] = teams_df
            else:
                teams_df = season_teams_cache[season]
            
            if teams_df is None or teams_df.empty:
                fixed_opponent_ids.append(None)
                fixed_opponent_names.append(None)
                fixed_opponent_difficulties.append(None)
                fixed_is_home.append(None)
                continue
            
            # Get player's team ID
            player_team = teams_df[teams_df['name'] == player_team_name]
            
            if player_team.empty:
                print(f"Row {idx}: Team '{player_team_name}' not found for season {season}")
                fixed_opponent_ids.append(None)
                fixed_opponent_names.append(None)
                fixed_opponent_difficulties.append(None)
                fixed_is_home.append(None)
                continue
                
            player_team_id = player_team['id'].values[0]
            
            # Find the match in fixtures
            if season not in match_dictionary or gameweek not in match_dictionary[season]:
                fixed_opponent_ids.append(None)
                fixed_opponent_names.append(None)
                fixed_opponent_difficulties.append(None)
                fixed_is_home.append(None)
                continue
                
            gameweek_matches = match_dictionary[season][gameweek]
            
            opponent_id = None
            opponent_name = None
            difficulty = None
            is_home = None
            
            for match in gameweek_matches:
                if match['team_h'] == player_team_id:
                    # Player's team is home
                    is_home = True
                    opponent_id = match['team_a']
                    difficulty = match['team_h_difficulty']
                    break
                elif match['team_a'] == player_team_id:
                    # Player's team is away
                    is_home = False
                    opponent_id = match['team_h']
                    difficulty = match['team_a_difficulty']
                    break
            
            # Get opponent name from team ID
            if opponent_id is not None:
                opponent_team = teams_df[teams_df['id'] == opponent_id]
                if not opponent_team.empty:
                    opponent_name = opponent_team['name'].values[0]
            
            fixed_opponent_ids.append(opponent_id)
            fixed_opponent_names.append(opponent_name)
            fixed_opponent_difficulties.append(difficulty)
            fixed_is_home.append(is_home)
            
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            fixed_opponent_ids.append(None)
            fixed_opponent_names.append(None)
            fixed_opponent_difficulties.append(None)
            fixed_is_home.append(None)
    
    df['Opponent ID'] = fixed_opponent_ids
    df['Opponent Name'] = fixed_opponent_names
    df['Opponent Difficulty'] = fixed_opponent_difficulties
    df['Is Home'] = fixed_is_home
    
    return df

In [9]:
# Load your training data
df = pd.read_csv("output/training_data.csv")
print(f"Loaded {len(df)} rows")

# %%
# Apply the fix
print("Fixing opponent data...")
df_fixed = fix_opponent_and_difficulty(df, data_folder='data')

# %%
# Verify the fix for Sokratis
fixed_row = df_fixed[
    (df_fixed['Player Name'] == 'Sokratis Papastathopoulos') & 
    (df_fixed['season'] == '2019-20') & 
    (df_fixed['Gameweek'] == 29)
]

print("\nFixed Training Data Row:")
print(fixed_row[['Player Name', 'Player Team Name', 'season', 'Gameweek', 
                 'Opponent Name', 'Opponent ID', 'Is Home', 'Opponent Difficulty']])


Loaded 168972 rows
Fixing opponent data...
Processing row 0/168972...
Processing row 1000/168972...
Processing row 2000/168972...
Processing row 3000/168972...
Processing row 4000/168972...
Processing row 5000/168972...
Processing row 6000/168972...
Processing row 7000/168972...
Processing row 8000/168972...
Processing row 9000/168972...
Processing row 10000/168972...
Processing row 11000/168972...
Processing row 12000/168972...
Processing row 13000/168972...
Processing row 14000/168972...
Processing row 15000/168972...
Processing row 16000/168972...
Processing row 17000/168972...
Processing row 18000/168972...
Processing row 19000/168972...
Processing row 20000/168972...
Processing row 21000/168972...
Processing row 22000/168972...
Processing row 23000/168972...
Processing row 24000/168972...
Processing row 25000/168972...
Processing row 26000/168972...
Processing row 27000/168972...
Processing row 28000/168972...
Processing row 29000/168972...
Processing row 30000/168972...
Processin

In [6]:

df_master = pd.read_csv("master_team_list.csv")
print(df_master.columns)

Index(['season', 'team', 'team_name'], dtype='object')


In [10]:
# Check for any remaining issues
print("\n=== Data Quality Check ===")
print(f"Rows with missing Opponent Name: {df_fixed['Opponent Name'].isna().sum()}")
print(f"Rows with missing Opponent ID: {df_fixed['Opponent ID'].isna().sum()}")
print(f"Rows with missing Opponent Difficulty: {df_fixed['Opponent Difficulty'].isna().sum()}")
print(f"Rows with missing Is Home: {df_fixed['Is Home'].isna().sum()}")



=== Data Quality Check ===
Rows with missing Opponent Name: 27
Rows with missing Opponent ID: 27
Rows with missing Opponent Difficulty: 27
Rows with missing Is Home: 27


In [11]:
# Save the corrected data
df_fixed.to_csv('output/training_data_fixed.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ Fixed file saved to: output/training_data_fixed.csv")
print(f"📊 Total rows: {len(df_fixed):,}")


✅ Fixed file saved to: output/training_data_fixed.csv
📊 Total rows: 168,972
